# A general L-shaped method

The companion notebook `two_stages.ipynb` writes **every** constraint as $\geq$, so that all the
second-stage multipliers share one sign. That is convenient, but it forces the modeller to rewrite
the problem before coding it, and an equality has to be split into two rows.

Here we keep the constraints as they are naturally written — $\leq$, $=$ or $\geq$, in **both**
stages, plus arbitrary bounds on the recourse variables — and let the algorithm deal with the signs.
Two ideas make this painless:

1. whatever the row senses, the convention JuMP uses for `dual` is *exactly* the one for which
   $Q(x,\xi) = \pi^T(h(\xi) - T(\xi)x)$, so no per-row sign correction is ever needed;
2. both cuts are built as **supporting hyperplanes** — a subgradient plus the requirement of being
   tight at $x^k$ — instead of using the closed-form intercept of the slides. The two coincide in
   the textbook setting, but only the first stays valid when the recourse variables carry finite
   bounds.

In [1]:
using JuMP
using LinearAlgebra
using Printf
using HiGHS

const SOLVER = HiGHS.Optimizer

HiGHS.Optimizer

## The problem class

$$
\min_{x \geq 0} \left\{ c^Tx + \mathbb{E}_{\boldsymbol{\xi}}[Q(x,\boldsymbol{\xi})] \ \middle|\
  Ax \lesseqgtr b \right\},
\qquad
Q(x,\xi) = \min_{l \leq y \leq u} \left\{ q^Ty \ \middle|\
  Wy \lesseqgtr h(\xi) - T(\xi)x \right\},
$$

where each row carries its own sense, given row by row. Only the first stage is required to be
non-negative; the recourse variables may have any bounds, the default $0 \leq y < +\infty$ being the
setting of the course.

In [2]:
"""Sense of a constraint row."""
@enum Sense LEQ EQ GEQ

"""Accept `'<'`, `'='`, `'>'` (or `"<="`, `"=="`, `">="`) as well as `Sense` values."""
to_sense(s::Sense) = s
to_sense(s::AbstractChar) = s in ('<', '≤') ? LEQ : s in ('>', '≥') ? GEQ :
                            s == '=' ? EQ : error("unknown sense: $s")
to_sense(s::AbstractString) = to_sense(first(s))

"""
    TwoStageProblem(; c, A, senses1, b, q, W, senses2, T, h, ξ, p, lb = 0, ub = Inf)

Two-stage stochastic linear program with arbitrary row senses in both stages.
`senses1` and `senses2` are vectors of `Sense` (or of `'<'` / `'='` / `'>'`), one per row of `A`
and of `W`. `T` and `h` are functions of the scenario, or constants. `lb` and `ub` are the bounds
on the recourse variables, either scalars or vectors.
"""
struct TwoStageProblem
    c::Vector{Float64}
    A::AbstractMatrix{Float64}
    senses1::Vector{Sense}
    b::Vector{Float64}
    q::Vector{Float64}
    W::AbstractMatrix{Float64}
    senses2::Vector{Sense}
    T::Function
    h::Function
    ξ::Vector
    p::Vector{Float64}
    lb::Vector{Float64}
    ub::Vector{Float64}
end

function TwoStageProblem(; c, A, senses1, b, q, W, senses2, T, h, ξ, p, lb = 0.0, ub = Inf)
    ny = length(q)
    pb = TwoStageProblem(
        Float64.(c), Float64.(A), to_sense.(collect(senses1)), Float64.(b),
        Float64.(q), Float64.(W), to_sense.(collect(senses2)),
        T isa AbstractMatrix ? (_ -> Float64.(T)) : T,
        h isa AbstractVector ? (_ -> Float64.(h)) : h,
        collect(ξ), Float64.(p),
        lb isa Number ? fill(Float64(lb), ny) : Float64.(lb),
        ub isa Number ? fill(Float64(ub), ny) : Float64.(ub))
    @assert size(pb.A, 2) == length(pb.c)       "A and c disagree on the number of columns"
    @assert size(pb.A, 1) == length(pb.b) == length(pb.senses1) "A, b and senses1 disagree"
    @assert size(pb.W, 2) == length(pb.q)       "W and q disagree on the number of columns"
    @assert size(pb.W, 1) == length(pb.senses2) "W and senses2 disagree on the number of rows"
    @assert length(pb.ξ) == length(pb.p)        "one probability per scenario is required"
    @assert isapprox(sum(pb.p), 1; atol = 1e-9) "the probabilities must sum up to one"
    for s in pb.ξ
        @assert size(pb.T(s)) == (size(pb.W, 1), length(pb.c)) "T(ξ) has a wrong size"
        @assert length(pb.h(s)) == size(pb.W, 1)               "h(ξ) has a wrong length"
    end
    return pb
end

n_x(pb::TwoStageProblem)         = length(pb.c)
n_y(pb::TwoStageProblem)         = length(pb.q)
n_rows(pb::TwoStageProblem)      = size(pb.W, 1)
n_scenarios(pb::TwoStageProblem) = length(pb.ξ)

n_scenarios (generic function with 1 method)

Adding a block of rows with mixed senses is the only place where the sense is looked at. Note that
the resulting vector of constraint references is **heterogeneous**: a $\leq$ row, an $=$ row and a
$\geq$ row live in three different MOI sets, so the container cannot be given a single concrete
type. Everything we do with it afterwards — `dual.`, `set_normalized_rhs.` — works all the same.

In [3]:
"""Add `lhs[i] ⋛ rhs[i]` to `m`, one sense per row, and return the constraint references."""
function add_rows!(m::Model, lhs::AbstractVector, senses::Vector{Sense}, rhs::AbstractVector)
    con = Vector{Any}(undef, length(senses))
    for i in eachindex(senses)
        con[i] = senses[i] == LEQ ? @constraint(m, lhs[i] <= rhs[i]) :
                 senses[i] == GEQ ? @constraint(m, lhs[i] >= rhs[i]) :
                                    @constraint(m, lhs[i] == rhs[i])
    end
    return con
end

"""Declare the recourse variables with their bounds (either of which may be infinite)."""
function recourse_variables!(m::Model, pb::TwoStageProblem)
    y = @variable(m, [1:n_y(pb)])
    for j in 1:n_y(pb)
        isfinite(pb.lb[j]) && set_lower_bound(y[j], pb.lb[j])
        isfinite(pb.ub[j]) && set_upper_bound(y[j], pb.ub[j])
    end
    return y
end

recourse_variables!

## Why no sign correction is needed

Write the recourse problem with a multiplier $\pi_i$ per row and form the Lagrangian. For a row
$W_i y \geq r_i$ we relax with $\lambda_i \geq 0$ and obtain the term $\lambda_i(r_i - W_iy)$; for a
row $W_i y \leq r_i$ we relax with $\mu_i \geq 0$ and obtain $\mu_i(W_iy - r_i)$, which is the
*same* term with $\pi_i = -\mu_i \leq 0$. In all three cases the Lagrangian reads

$$L(y, \pi) = \pi^Tr + (q - W^T\pi)^Ty,$$

so the dual is, uniformly,

$$Q(x,\xi) = \max_{\pi \in \Pi} \left\{ \pi^T(h(\xi) - T(\xi)x) \ \middle|\ W^T\pi \leq q \right\},
\qquad
\Pi = \left\{ \pi \ \middle|\
\begin{array}{ll}
\pi_i \geq 0 & \text{on a } \geq \text{ row} \\
\pi_i \leq 0 & \text{on a } \leq \text{ row} \\
\pi_i \text{ free} & \text{on an } = \text{ row}
\end{array} \right\}.$$

That sign pattern is **exactly** what JuMP returns for a minimization problem. In other words, the
convention of the solver and the convention of the course agree, and `dual.(con)` can be used as
is, whatever the mix of senses. What the previous notebook warned about was never the sense of a
row by itself, but the risk of writing a row in one form and $(W, T, h)$ in another.

## Cuts as supporting hyperplanes

The only place where $x$ enters the second stage is the right-hand side $h(\xi) - T(\xi)x$, so
$-T(\xi)^T\pi$ is a subgradient of $Q(\cdot,\xi)$ at $x^k$, and

$$E = \sum_s p_s\, T(\xi_s)^T\pi_s \quad\Longrightarrow\quad -E \in \partial \mathcal{Q}(x^k).$$

The optimality cut is the supporting hyperplane of $\mathcal{Q}$ at $x^k$ with that slope, i.e.
$\theta \geq \mathcal{Q}(x^k) - E^T(x - x^k)$, which we write

$$E^Tx + \theta \geq e, \qquad e = \mathcal{Q}(x^k) + E^Tx^k.$$

With $l = 0$ and $u = +\infty$, strong duality gives $\mathcal{Q}(x^k) = \sum_s p_s\pi_s^T(h_s - T_sx^k)$
and $e$ collapses to the familiar $e = \sum_s p_s h_s^T\pi_s$ of the slides. **With a finite bound
on a recourse variable this is no longer true**: the dual objective then also carries the bound
terms, $Q = \pi^Tr + \mu_l^Tl - \mu_u^Tu$, and $\sum_s p_s h_s^T\pi_s$ over-estimates $e$ whenever a
bound is active, producing a cut that chops off the optimum. Computing $e$ from tightness at $x^k$
costs nothing and is immune to this.

The feasibility cut is obtained in the same way from the elastic problem below: if $v_s(x)$ denotes
its optimal value — the smallest total infeasibility of scenario $s$ at $x$ — then $v_s$ is convex,
$-T(\xi_s)^T\sigma$ is a subgradient of it, feasibility means $v_s(x) \leq 0$, and linearizing at
$x^k$ gives

$$E_f^Tx \geq e_f, \qquad E_f = T(\xi_s)^T\sigma, \qquad e_f = v_s(x^k) + E_f^Tx^k,$$

which again reduces to $\sigma^Th(\xi_s)$ in the textbook case.

In [4]:
"""
Elastic version of the second stage: one artificial variable per inequality row, two per equality
row, so that the problem is always feasible and its optimal value is the smallest total violation.
"""
function elastic_stage(pb::TwoStageProblem; optimizer = SOLVER)
    m = Model(optimizer)
    set_silent(m)
    y = recourse_variables!(m, pb)
    @variable(m, wp[1:n_rows(pb)] >= 0)      # violation upwards
    @variable(m, wn[1:n_rows(pb)] >= 0)      # violation downwards
    lhs = pb.W * y
    con = Vector{Any}(undef, n_rows(pb))
    for i in 1:n_rows(pb)
        con[i] = pb.senses2[i] == GEQ ? @constraint(m, lhs[i] + wp[i] >= 0.0) :
                 pb.senses2[i] == LEQ ? @constraint(m, lhs[i] - wn[i] <= 0.0) :
                                        @constraint(m, lhs[i] + wp[i] - wn[i] == 0.0)
    end
    @objective(m, Min, sum(wp) + sum(wn))
    return m, y, con
end

elastic_stage

## The building blocks

As before, the recourse and elastic models are built once and only their right-hand side is updated.

In [5]:
"""A model whose right-hand side is updated in place; `con` is heterogeneous by construction."""
struct StageProblem
    model::Model
    y::Vector{VariableRef}
    con::Vector{Any}
end

"""Recourse problem min qᵀy s.t. Wy ⋛ ⋅, l ≤ y ≤ u."""
function second_stage(pb::TwoStageProblem; optimizer = SOLVER)
    m = Model(optimizer)
    set_silent(m)
    y = recourse_variables!(m, pb)
    con = add_rows!(m, pb.W * y, pb.senses2, zeros(n_rows(pb)))
    @objective(m, Min, dot(pb.q, y))
    return StageProblem(m, y, con)
end

function elastic_problem(pb::TwoStageProblem; optimizer = SOLVER)
    m, y, con = elastic_stage(pb; optimizer)
    return StageProblem(m, y, con)
end

"""Right-hand side of the second stage at (x, ξ)."""
rhs(pb::TwoStageProblem, x, ξ) = pb.h(ξ) - pb.T(ξ) * x

"""Update the right-hand side and re-optimize; returns the termination status."""
function solve_stage!(sp::StageProblem, pb::TwoStageProblem, x, ξ)
    set_normalized_rhs.(sp.con, rhs(pb, x, ξ))
    optimize!(sp.model)
    return termination_status(sp.model)
end

solve_stage!

In [6]:
"""Extensive form (deterministic equivalent), for reference."""
function extensive_form(pb::TwoStageProblem; optimizer = SOLVER)
    S = n_scenarios(pb)
    m = Model(optimizer)
    set_silent(m)
    @variable(m, x[1:n_x(pb)] >= 0)
    y = [recourse_variables!(m, pb) for _ in 1:S]
    add_rows!(m, pb.A * x, pb.senses1, pb.b)
    for s in 1:S
        add_rows!(m, pb.T(pb.ξ[s]) * x + pb.W * y[s], pb.senses2, pb.h(pb.ξ[s]))
    end
    @objective(m, Min, dot(pb.c, x) + sum(pb.p[s] * dot(pb.q, y[s]) for s in 1:S))
    optimize!(m)
    @assert termination_status(m) == MOI.OPTIMAL "extensive form: $(termination_status(m))"
    return m, value.(x), objective_value(m)
end

extensive_form

## The algorithm

The loop itself is unchanged with respect to the previous notebook; only the way the two cuts are
assembled differs.

In [7]:
"""
    lshaped(pb; maxiter, tol, feastol, verbose)

Single-cut L-shaped method for a `TwoStageProblem` with arbitrary row senses.
"""
function lshaped(pb::TwoStageProblem; optimizer = SOLVER, maxiter = 200,
                 tol = 1e-8, feastol = 1e-7, verbose = true)
    n = n_x(pb)

    master = Model(optimizer)
    set_silent(master)
    @variable(master, x[1:n] >= 0)
    @variable(master, θ)
    add_rows!(master, pb.A * x, pb.senses1, pb.b)
    @objective(master, Min, dot(pb.c, x))     # θ joins the objective with the first optimality cut

    recourse = second_stage(pb; optimizer)
    elastic = elastic_problem(pb; optimizer)

    n_opt = n_feas = 0
    history = NamedTuple[]
    verbose && println(" iter   lower bound   upper bound             θ          Q(x)")

    for k in 1:maxiter
        optimize!(master)
        termination_status(master) == MOI.OPTIMAL ||
            error("master problem: $(termination_status(master))")
        xk = value.(x)
        θk = n_opt > 0 ? value(θ) : -Inf
        lb = n_opt > 0 ? objective_value(master) : -Inf

        Q, E, cut_added = 0.0, zeros(n), false
        for s in 1:n_scenarios(pb)
            ξs = pb.ξ[s]
            status = solve_stage!(recourse, pb, xk, ξs)

            if status != MOI.OPTIMAL
                # either infeasible, or the solver could not tell: the elastic problem decides
                solve_stage!(elastic, pb, xk, ξs) == MOI.OPTIMAL ||
                    error("elastic problem, scenario $s: $(termination_status(elastic.model))")
                v = objective_value(elastic.model)
                v > feastol || error("scenario $s: recourse feasible but status $status; " *
                                     "the second stage is probably unbounded")
                σ = dual.(elastic.con)
                Ef = pb.T(ξs)' * σ
                @constraint(master, dot(Ef, x) >= v + dot(Ef, xk))   # tight at xᵏ
                n_feas += 1
                cut_added = true
                verbose && @printf("%5d   feasibility cut (scenario %d, violation %.4g)\n", k, s, v)
                break
            end

            π = dual.(recourse.con)     # sign pattern already matches Q = πᵀ(h - Tx)
            Q += pb.p[s] * objective_value(recourse.model)
            E .+= pb.p[s] .* (pb.T(ξs)' * π)
        end
        cut_added && continue

        ub = dot(pb.c, xk) + Q
        push!(history, (iteration = k, lb = lb, ub = ub, θ = θk, Q = Q, x = xk))
        if verbose
            lb_str = n_opt > 0 ? @sprintf("%.6f", lb) : "-Inf"
            θ_str = n_opt > 0 ? @sprintf("%.6f", θk) : "-Inf"
            @printf("%5d   %11s   %11.6f   %11s   %11.6f\n", k, lb_str, ub, θ_str, Q)
        end

        if θk >= Q - tol
            verbose && @printf("converged in %d iterations (%d optimality cuts, %d feasibility cuts)\n",
                               k, n_opt, n_feas)
            return (x = xk, objective = ub, iterations = k, optimality_cuts = n_opt,
                    feasibility_cuts = n_feas, master = master, history = history)
        end

        @constraint(master, dot(E, x) + θ >= Q + dot(E, xk))         # tight at xᵏ
        if n_opt == 0
            @objective(master, Min, dot(pb.c, x) + θ)                # θ is now bounded below
        end
        n_opt += 1
    end
    error("no convergence in $maxiter iterations")
end

lshaped

## Instance 1: the ice-cream problem, written naturally

This is the instance of `two_stages.ipynb`, but with the constraints left as they were stated: the
capacity rows are $\leq$, the demand rows are $\geq$, and the budget is a $\leq$ row of the first
stage. No row has to be reversed, and $T$ carries the sign the model dictates
($\sum_j y_{ij} \leq x_i$ reads $W_iy \leq h_i - T_ix$ with $h_i = 0$ and $T_{ii} = -1$).

In [8]:
const NPLANTS = 4
const NFLAVORS = 3
opening_costs = [10.0, 7.0, 16.0, 6.0]
production_costs = [40.0 24.0 4.0; 45.0 27.0 4.5; 32.0 19.2 3.2; 55.0 33.0 5.5]

function icecream(; min_capacity = 12.0, budget = 120.0, ub = Inf)
    ny = NPLANTS * NFLAVORS
    idx(i, j) = NFLAVORS * (i - 1) + j
    W = zeros(NPLANTS + NFLAVORS, ny)
    T = zeros(NPLANTS + NFLAVORS, NPLANTS)
    for i in 1:NPLANTS, j in 1:NFLAVORS
        W[i, idx(i, j)] = 1.0                 # Σ_j y_ij ≤ x_i
        W[NPLANTS + j, idx(i, j)] = 1.0       # Σ_i y_ij ≥ d_j(ξ)
    end
    for i in 1:NPLANTS
        T[i, i] = -1.0
    end
    return TwoStageProblem(
        c = opening_costs,
        A = [ones(1, NPLANTS); opening_costs'], senses1 = ['>', '<'],
        b = [min_capacity, budget],
        q = vec(permutedims(production_costs)),
        W = W, senses2 = vcat(fill('<', NPLANTS), fill('>', NFLAVORS)),
        T = T, h = ξ -> vcat(zeros(NPLANTS), ξ, [3.0, 2.0]),
        ξ = [3.0, 5.0, 7.0], p = [0.3, 0.4, 0.3], ub = ub)
end

pb1 = icecream()
_, x_ef, obj_ef = extensive_form(pb1)
@printf("extensive form: %.6f   x = %s\n", obj_ef, string(x_ef))

extensive form: 381.853333   x = [2.666666666666666, 4.0, 3.3333333333333335, 2.0]


In [9]:
res1 = lshaped(pb1)
println()
@printf("x = %s\nobjective %.6f (extensive form %.6f)\n", string(res1.x), res1.objective, obj_ef)

 iter   lower bound   upper bound             θ          Q(x)
    1          -Inf    457.000000          -Inf    385.000000
    2    325.000000    400.000000    205.000000    280.000000
    3    362.500000    397.950000    280.000000    315.450000
    4    374.828179    388.499725    254.828179    268.499725
    5    377.154730    383.478647    257.154730    263.478647
    6    379.212091    383.834458    259.212091    263.834458
    7    380.012666    382.752596    260.012666    262.752596
    8    381.452879    382.366208    261.452879    262.366208
    9    381.716695    382.103792    261.716695    262.103792
   10    381.853333    381.853333    261.853333    261.853333
converged in 10 iterations (9 optimality cuts, 0 feasibility cuts)

x = [2.666666666666506, 4.000000000000057, 3.3333333333333917, 2.0000000000000453]
objective 381.853333 (extensive form 381.853333)


We recover $x^\star = (8/3,\ 4,\ 10/3,\ 2)$ and $28639/75 \approx 381.8533$, the value obtained in
`two_stages.ipynb` from the all-$\geq$ formulation: rewriting the rows was a convenience, not a
necessity.

Relaxing the minimum capacity again lets the master answer $x = 0$ and triggers feasibility cuts,
now generated from an elastic problem that mixes $\leq$ and $\geq$ rows.

In [10]:
res1b = lshaped(icecream(min_capacity = 0.0))
println()
@printf("x = %s, objective %.6f, %d feasibility cuts\n",
        string(res1b.x), res1b.objective, res1b.feasibility_cuts)

 iter   lower bound   upper bound             θ          Q(x)
    1   feasibility cut (scenario 1, violation 8)
    2   feasibility cut (scenario 2, violation 2)
    3   feasibility cut (scenario 3, violation 2)
    4          -Inf    457.000000          -Inf    385.000000
    5    325.000000    400.000000    205.000000    280.000000
    6    362.500000    397.950000    280.000000    315.450000
    7    374.828179    388.499725    254.828179    268.499725
    8    377.154730    383.478647    257.154730    263.478647
    9    379.212091    383.834458    259.212091    263.834458
   10    380.012666    382.752596    260.012666    262.752596
   11    381.452879    382.366208    261.452879    262.366208
   12    381.716695    382.103792    261.716695    262.103792
   13    381.853333    381.853333    261.853333    261.853333
converged in 13 iterations (9 optimality cuts, 3 feasibility cuts)

x = [2.666666666666559, 4.000000000000053, 3.3333333333333717, 2.0000000000000147], objective 381.85

## Instance 2: equality rows — the newsvendor

Order $x$ at unit cost $c = 5$, sell at $p = 12$ what the demand $\xi$ absorbs, salvage the rest at
$r = 2$. With $y = (s, v, u)$ — sold, salvaged, unmet — the second stage is made of two **equality**
rows,

$$s + v = x \quad\text{(everything ordered is sold or salvaged)}, \qquad
  s + u = \xi \quad\text{(the demand is met or lost)},$$

with cost $-p\,s - r\,v$. The first stage carries a $\leq$ row, the ordering capacity.

In [11]:
newsvendor = TwoStageProblem(
    c = [5.0],
    A = reshape([1.0], 1, 1), senses1 = ['<'], b = [100.0],       # x ≤ 100
    q = [-12.0, -2.0, 0.0],                                       # sold, salvaged, unmet
    W = [1.0 1.0 0.0; 1.0 0.0 1.0], senses2 = ['=', '='],
    T = reshape([-1.0, 0.0], 2, 1),
    h = ξ -> [0.0, ξ],
    ξ = [10.0, 20.0, 30.0, 40.0], p = [0.15, 0.35, 0.35, 0.15])

_, x_nv, obj_nv = extensive_form(newsvendor)
res2 = lshaped(newsvendor)
println()
@printf("extensive form: x = %.4f, value %.4f\n", x_nv[1], obj_nv)
@printf("L-shaped      : x = %.4f, value %.4f\n", res2.x[1], res2.objective)

 iter   lower bound   upper bound             θ          Q(x)
    1          -Inf      0.000000          -Inf      0.000000
    2   -700.000000     50.000000   -1200.000000   -450.000000
    3   -175.000000   -135.000000   -300.000000   -260.000000
    4   -151.000000   -140.500000   -316.000000   -305.500000
    5   -145.000000   -145.000000   -295.000000   -295.000000
converged in 5 iterations (4 optimality cuts, 0 feasibility cuts)

extensive form: x = 30.0000, value -145.0000
L-shaped      : x = 30.0000, value -145.0000


The critical ratio of the newsvendor is $(p-c)/(p-r) = 7/10$, and the smallest scenario whose
cumulative probability reaches it is $30$ (cumulative $0.85$, against $0.5$ at $20$): the optimal
order is $x^\star = 30$, for an expected profit of $145$, i.e. an optimal value of $-145$ for our
minimization. Both methods return it.

## Instance 3: bounds on the recourse variables

Suppose no plant can devote more than 3 units of capacity to a single flavor, i.e. $y_{ij} \leq 3$.
The bound is active at the optimum, which moves to $x^\star = (3, 4, 3, 2)$ with value
$1911/5 = 382.2$, and infeasible iterates now appear even though the total capacity is sufficient.

In [12]:
pb3 = icecream(ub = 3.0)
_, x3_ef, obj3_ef = extensive_form(pb3)
res3 = lshaped(pb3)
println()
@printf("extensive form: %.6f   x = %s\n", obj3_ef, string(x3_ef))
@printf("L-shaped      : %.6f   x = %s   (%d feasibility cuts)\n",
        res3.objective, string(res3.x), res3.feasibility_cuts)

 iter   lower bound   upper bound             θ          Q(x)
    1   feasibility cut (scenario 2, violation 2)
    2   feasibility cut (scenario 3, violation 2)
    3   feasibility cut (scenario 3, violation 1)
    4          -Inf    420.200000          -Inf    341.200000
    5   feasibility cut (scenario 2, violation 2)
    6   feasibility cut (scenario 3, violation 2)
    7    337.533333    394.666667    217.533333    274.666667
    8    362.565392    390.263783    242.565392    270.263783
    9    370.622681    390.518668    250.622681    270.518668
   10    373.798048    382.748323    253.798048    262.748323
   11   feasibility cut (scenario 3, violation 0.6009)
   12    377.353265    394.978931    282.842701    300.468367
   13    378.469846    388.990487    274.308542    284.829183
   14    381.151451    386.457359    263.445179    268.751087
   15    381.412805    384.594673    268.563095    271.744963
   16    381.806323    382.681812    265.177530    266.053019
   17    382.

### Why the textbook intercept fails here

At a point where a bound is active, $\sum_s p_s h_s^T\pi_s$ and $\mathcal{Q}(x) + E^Tx$ no longer
agree: strong duality now reads $Q = \pi^Tr - \mu_u^Tu$ (with $l = 0$), so the first expression
exceeds the second by $\sum_s p_s \mu_{u,s}^Tu \geq 0$. Using it would impose
$\theta \geq \mathcal{Q}(x^k) + \text{something positive}$ at $x^k$ — a cut that removes the
optimum.

In [13]:
"""Compare the two ways of computing the intercept of the optimality cut at `x`."""
function compare_intercepts(pb::TwoStageProblem, x)
    sp = second_stage(pb)
    Q, E, e_textbook = 0.0, zeros(n_x(pb)), 0.0
    for s in 1:n_scenarios(pb)
        solve_stage!(sp, pb, x, pb.ξ[s]) == MOI.OPTIMAL || return nothing
        π = dual.(sp.con)
        Q += pb.p[s] * objective_value(sp.model)
        E .+= pb.p[s] .* (pb.T(pb.ξ[s])' * π)
        e_textbook += pb.p[s] * dot(π, pb.h(pb.ξ[s]))
    end
    return (Q = Q, e_tight = Q + dot(E, x), e_textbook = e_textbook)
end

x_probe = fill(4.0, NPLANTS)
for (label, pb) in (("no bound   ", icecream()), ("y ≤ 2      ", icecream(ub = 2.0)))
    r = compare_intercepts(pb, x_probe)
    @printf("%s Q(x) = %9.4f   e (tight) = %10.4f   e (textbook) = %10.4f   difference = %8.4f\n",
            label, r.Q, r.e_tight, r.e_textbook, r.e_textbook - r.e_tight)
end

no bound    Q(x) =  251.4600   e (tight) =   288.4200   e (textbook) =   288.4200   difference =   0.0000
y ≤ 2       Q(x) =  264.2500   e (tight) =   270.2500   e (textbook) =   323.2500   difference =  53.0000


Without bounds the two coincide, as the slides say. With an active bound the textbook expression is
strictly larger, and the corresponding cut would be invalid. This is the reason the algorithm above
never uses it.

## Summary

* Row senses are data, not something to be normalized away: `senses1` and `senses2` are read once,
  in `add_rows!` and in the elastic problem.
* Because JuMP's dual convention for a minimization coincides with the sign pattern of $\Pi$, the
  multipliers can be used as they come, whatever the mix of $\leq$, $=$ and $\geq$.
* Building both cuts from a subgradient and tightness at $x^k$ keeps them valid beyond the
  textbook setting — in particular when the recourse variables are bounded.
* The value-of-information quantities of `two_stages.ipynb` (`EVPI`, `VSS`) transpose to this
  structure unchanged; only the model builders have to be replaced.